# 15 — Full Retraining Reference for Profile-Memory Unlearning

**Purpose:** train a fresh Qwen profile-memory model as if the 100 forget recipients had never been included.

It follows the completed Notebook 14 recipe exactly: same Qwen3-4B base, same LoRA, same 14 main factual fields, same six-field reinforcement, and the same training settings. The only change is that it trains on the 200 retain recipients only.

## Pipeline

1. Load or recreate the exact Notebook 14 recipient groups.
2. Mark the first 100 memory recipients as the forget group.
3. Train a fresh model on the remaining 200 retain recipients only.
4. Save the full-retraining reference model and runtime.

This notebook does not change your completed original model.

In [ ]:
%pip install -q -U unsloth trl datasets scikit-learn

from pathlib import Path
import json
import sys
import pandas as pd
import torch

REPO_OVERRIDE = None
repo_candidates = [Path('/content/qub-machine-unlearning'), Path.cwd(), Path.cwd().parent]
if REPO_OVERRIDE:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))
REPO_ROOT = next((path for path in repo_candidates if (path / 'code' / 'final_submission').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Clone the repository into /content, then rerun this cell.')
sys.path.insert(0, str(REPO_ROOT / 'code' / 'final_submission' / 'notebooks'))
from profile_memory_unlearning_common import *

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required.')
set_seed()
print('GPU:', torch.cuda.get_device_name(0))
print('Repository:', REPO_ROOT)

In [ ]:
PROFILES = load_profiles(REPO_ROOT)
CONTRACT = load_or_recreate_groups(REPO_ROOT, PROFILES)
PATHS = paths(REPO_ROOT)

display(pd.Series({
    'Forget recipients': len(CONTRACT['forget_recipient_ids']),
    'Retain recipients': len(CONTRACT['retain_recipient_ids']),
    'Unseen controls': len(CONTRACT['control_recipient_ids']),
    'Group source': CONTRACT['source'],
}).to_frame('Value'))

## Train the full-retraining reference

The fresh model receives no profile questions for the forget recipients.

In [ ]:
model, tokenizer = load_base_model()
main_examples = qa_examples(
    PROFILES,
    CONTRACT['retain_recipient_ids'],
    CONTRACT['memory_fields'],
    tokenizer,
)
focused_examples = qa_examples(
    PROFILES,
    CONTRACT['retain_recipient_ids'],
    CONTRACT['focused_fields'],
    tokenizer,
)
assert len(main_examples) == 2_800
assert len(focused_examples) == 1_200

main_stats = train_sft(
    model, tokenizer, main_examples,
    PATHS['artifacts'] / 'full_retraining_main_stage',
    epochs=20, batch_size=16, learning_rate=1e-4, schedule='cosine',
)
focused_stats = train_sft(
    model, tokenizer, focused_examples,
    PATHS['artifacts'] / 'full_retraining_focused_stage',
    epochs=20, batch_size=32, learning_rate=1e-4, schedule='constant',
)
FULL_RETRAIN_DIR = PATHS['artifacts'] / 'full_retrained_profile_memory_without_forget'
FULL_RETRAIN_ARCHIVE = save_adapter(model, tokenizer, FULL_RETRAIN_DIR)
pd.DataFrame([
    {'stage': 'main', **main_stats},
    {'stage': 'focused', **focused_stats},
]).to_csv(PATHS['results'] / 'full_retraining_runtime.csv', index=False)
print('Saved full-retraining archive:', FULL_RETRAIN_ARCHIVE)

In [ ]:
checks = {
    'Fresh full-retrained model saved': FULL_RETRAIN_DIR.exists(),
    'Full-retrained archive saved': FULL_RETRAIN_ARCHIVE.exists(),
    'Forget group absent from main training': not set(CONTRACT['forget_recipient_ids']).intersection(main_examples['recipient_id']),
    'Forget group absent from focused training': not set(CONTRACT['forget_recipient_ids']).intersection(focused_examples['recipient_id']),
    'Runtime saved': (PATHS['results'] / 'full_retraining_runtime.csv').exists(),
}
display(pd.Series(checks).to_frame('Pass'))
assert all(checks.values())